# Aegis Quantum-Cognitive — Investor Interactive Demo

**AQC** packages a machine-readable **Advanced Knowledge Base (AKB)** —
bio-cyber ontology, named threat profiles (including **Slater HNDL 2026**),
and regulatory mapping — that feeds CBOM and HNDL engines instead of magic numbers.

Run this notebook on **Google Colab** or locally. *Not legal, investment, or medical advice.*


In [ ]:
# Colab-friendly deps (no liboqs required for this notebook)
import sys, subprocess
for pkg in ("pyyaml", "networkx", "matplotlib", "numpy", "gymnasium"):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])


## 1) Clone AQC (Colab) or use cwd (local)

In Colab, uncomment the clone. Locally, `cd` into the repo root.


In [ ]:
import os, sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules or os.path.exists("/content")
if IN_COLAB:
    REPO = Path("/content/Aegis_Q_Cognitive")
    if not REPO.joinpath("src/aqc").is_dir():
        import subprocess
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/AAH20/Aegis_Q_Cognitive.git", str(REPO)],
            check=True,
        )
    os.chdir(REPO)
else:
    REPO = Path.cwd().resolve()

sys.path.insert(0, str(REPO / "src"))
print("Repo:", REPO)


## 2) Knowledge graph (ontology + mandates)

Nodes = concepts; edges = "informs" / "mandates". This is **analytic**, not a clinical device.


In [ ]:
import yaml
import networkx as nx
import matplotlib.pyplot as plt
from pathlib import Path

KB = REPO / "knowledge_base"
if not KB.is_dir():
    KB = REPO / "src/aqc/knowledge_base"

with open(KB / "bio_cyber_ontology/neural_threats.yaml", encoding="utf-8") as f:
    neural = yaml.safe_load(f)
with open(KB / "compliance_mappings/nsm10_cnsa2.yaml", encoding="utf-8") as f:
    comp = yaml.safe_load(f)

G = nx.DiGraph()
G.add_node("Soul Catcher 2.0", layer="threat")
G.add_node("HNDL stream fingerprint", layer="signal")
G.add_node("NSM-10 migration", layer="policy")
G.add_node("CNSA 2.0 suite", layer="policy")
G.add_node("FIPS 203 ML-KEM", layer="crypto")
G.add_node("FIPS 204 ML-DSA", layer="crypto")

_fp = neural["soul_catcher_2_0"]["hndl_stream_fingerprint"]
assert _fp["entropy_ciphertext_floor_bits_per_byte"] > 0
G.add_edge("Soul Catcher 2.0", "HNDL stream fingerprint")
G.add_edge("HNDL stream fingerprint", "NSM-10 migration")

for m in comp["mandates"].values():
    doc = m["document"]
    G.add_node(doc, layer="policy")
    G.add_edge(doc, "FIPS 203 ML-KEM")

plt.figure(figsize=(10, 6))
pos = nx.spring_layout(G, seed=42)
colors = {"threat": "#c0392b", "signal": "#2980b9", "policy": "#27ae60", "crypto": "#8e44ad"}
nc = [colors.get(G.nodes[n]["layer"], "#95a5a6") for n in G.nodes]
nx.draw(G, pos, with_labels=True, node_color=nc, font_size=9, arrows=True)
plt.title("AQC Knowledge Base — excerpt (demo layout)")
plt.axis("off")
plt.tight_layout()
plt.show()


## 3) Simulated Interrogation AGI vs "Target CEO Neuralink-class" traffic

**Story:** classical tunnels degrade under stress; **ML-KEM-class** posture stays in the green **in this toy model** (not a penetration test).


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

try:
    from aqc.agi_interrogator.recursive_agent import RecursiveInterrogator
except ImportError:
    raise SystemExit("Install agi extra: pip install -e '.[agi]' from repo root (needs gymnasium).")

env = JADC2BioEnv(target="colab/ceo-neuralink-class-sim", seed=42)
agent = RecursiveInterrogator(env=env, plateau_window=5, stuck_epochs=50)

epochs = 25
classical_score = []
pqc_score = []
for ep in range(epochs):
    np.random.seed(ep)
    classical_score.append(100 - ep * 3.2 - np.random.uniform(0, 4))
    pqc_score.append(98 - ep * 0.15 - np.random.uniform(0, 1.5))

# Short RL roll for reward curve
from collections import deque
log: deque[str] = deque(maxlen=200)
res = agent.run(epochs, log)
rewards = np.linspace(0, res.total_reward, 40)

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(classical_score, label="Classical ECDHE tunnel (model)", color="#c0392b")
ax[0].plot(pqc_score, label="ML-KEM-768 hybrid posture (model)", color="#27ae60")
ax[0].set_title("Toy decryption-stress index (demo only)")
ax[0].set_xlabel("epoch")
ax[0].legend()

ax[1].plot(rewards, color="#8e44ad")
ax[1].set_title(f"Interrogation AGI cumulative reward (sim), profile={env.target}")
ax[1].set_xlabel("step index (interpolated)")
plt.tight_layout()
plt.show()

print("Mutation generations:", res.mutation_generations, "| breach:", res.soul_catcher_breach)


## Why this matters to allocators (one paragraph)

Regulators are forcing **crypto inventory** and **PQC migration** artifacts onto MedTech and NSS suppliers. AQC turns those artifacts into **data**: CBOM, HNDL findings, and mapping rows consumed from an explicit **ontology**. The Interrogation AGI demo is **range-simulation** for continuous assurance narratives — scope it honestly in diligence.
